# 13 Packing: many subcircuits at once on one QPU

<a href="https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/13_packing.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a> <a href="https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/13_packing.ipynb"><img src="https://mybinder.org/badge_logo.svg" alt="Open in Binder"></a>

A device usually has many more qubits than one subcircuit uses. `qd.pack(circuits, backend)` compiles each circuit onto its own group of connected, well calibrated device qubits, with idle qubits between groups, and puts several in one device-wide circuit. So one job and one set of shots runs all of them.

This notebook needs no device or credentials. It uses a **fake backend**, IBM's published calibration snapshot of a 16-qubit device, simulated with Aer. No job goes to any QPU, and the noisy numbers below are a simulation with that snapshot's noise model.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qiskit]>=2.0"

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeGuadalupeV2
import Qdislib as qd

backend = FakeGuadalupeV2()
print(backend.name, backend.num_qubits, "qubits")


def example_circuit():
    """Six qubits in a line."""
    qc = QuantumCircuit(6)
    for q, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3, 0.7]):
        qc.ry(angle, q)
    for q in range(5):
        qc.cz(q, q + 1)
    for q, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9, 0.4]):
        qc.rx(angle, q)
    return qc


circuit = example_circuit()
cut = ["CZ_9"]
exact = qd.analytical_solution(circuit, "Z" * 6)
print(f"exact {exact:+.4f}")

fake_guadalupe 16 qubits
exact +0.1931


## Pack

Start from the list's circuits, with cut points deferred to the end and every qubit measured. Each circuit gets a group of `size` device qubits, by default the width of the widest. Groups are at least `buffer` idle qubits apart.

In [3]:
subs = qd.circuit_cutting_subcircuits(circuit, cut, "Z" * 6)
circuits = []
for qc in subs.circuits("qiskit", defer_measurements=True):
    measured = qc.copy()
    measured.measure_all()
    circuits.append(measured)

packed = qd.pack(circuits, backend, max_error=0.1, max_readout_error=0.2)
print(packed)
print("device-wide circuits:", len(packed.circuits), "of", packed.circuits[0].num_qubits, "qubits")
print("the first groups of qubits:", packed.regions[:3])
print("where each circuit went (packed circuit, register, device qubits):")
for slot in packed.slots[:4]:
    print("  ", slot)
print("two-qubit gates after compiling:", packed.two_qubit_gates)

PackedCircuits(12 circuits in 6 device circuits, 2 groups of qubits)
device-wide circuits: 6 of 16 qubits
the first groups of qubits: [(4, 6, 7, 10), (5, 8, 9, 11)]
where each circuit went (packed circuit, register, device qubits):
   (0, 'p0', (4, 6, 7, 10))
   (0, 'p1', (5, 8, 9, 11))
   (1, 'p0', (4, 6, 7, 10))
   (1, 'p1', (5, 8, 9, 11))
two-qubit gates after compiling: [2, 2, 2, 2, 6, 2, 6, 2, 2, 4, 2, 4]


`max_error` and `max_readout_error` exclude couplers and qubits worse than that. The defaults (0.02 and 0.05) suit current devices; this old snapshot is noisier, so they are raised here.

## Run and unpack

Run `packed.circuits` as they are: they are already compiled for the device. On a real device each is one plain job, as in the commented lines. `unpack` takes the results and returns each circuit's counts, keyed as if it had run alone, ready for `reconstruct_counts`.

```python
# On a real device (not run here)
from qiskit_ibm_runtime import SamplerV2
results = [SamplerV2(mode=backend).run([c], shots=4096).result()[0] for c in packed.circuits]
```

In [4]:
ideal = AerSimulator(method="statevector", seed_simulator=1)
counts = [ideal.run(c, shots=8192).result().get_counts() for c in packed.circuits]

value = subs.reconstruct_counts(packed.unpack(counts), bit_order="q0_last")
print(f"packed, no noise: {value:+.4f}   exact {exact:+.4f}")
print(f"{len(circuits)} circuits in {len(packed.circuits)} simulated jobs")

packed, no noise: +0.1934   exact +0.1931
12 circuits in 6 simulated jobs


## With the snapshot's noise

The same packed circuits with the noise model of the calibration snapshot, and each circuit alone on the qubits it had when packed, with `regions=`, to separate the effect of running side by side from the device's noise.

Comparing a few circuits alone before trusting a packed run is what we recommend on hardware too. So far packing has been checked on one real device (ibm_kobe), with shallow 3-5-qubit circuits, where packed and alone gave the same fidelity. Deeper circuits may show more crosstalk. How groups are chosen may also change between versions; `regions=` fixes them.

An Aer noise model has no crosstalk between groups, so **this simulation cannot show crosstalk**. It only checks that the code path works.

In [5]:
noisy = AerSimulator.from_backend(backend, seed_simulator=1)
noisy_counts = [noisy.run(c, shots=4096).result().get_counts() for c in packed.circuits]
packed_value = subs.reconstruct_counts(packed.unpack(noisy_counts), bit_order="q0_last")

alone_counts = []
for circuit_i, slot in zip(circuits, packed.slots):
    alone = qd.pack([circuit_i], backend, regions=[slot[2]], max_error=0.1, max_readout_error=0.2)
    run = noisy.run(alone.circuits[0], shots=4096).result().get_counts()
    alone_counts.append(alone.unpack([run])[0])
alone_value = subs.reconstruct_counts(alone_counts, bit_order="q0_last")

print(f"exact        {exact:+.4f}")
print(f"packed       {packed_value:+.4f}  in {len(packed.circuits)} jobs")
print(f"each alone   {alone_value:+.4f}  in {len(circuits)} jobs")

exact        +0.1931
packed       +0.1260  in 6 jobs
each alone   +0.1222  in 12 jobs


## Built into the IBM backend

With `pack=True` in its config, the IBM backend does all of the above. Each group of `circuits_per_task` subcircuits runs side by side, in plain jobs. A group holds one fragment's circuits. Choose `circuits_per_task` near the number of groups of qubits the device has for them, which `qd.pack(...).regions` shows. `pack={...}` passes options to `qd.pack`.

**Not run here:** it needs IBM credentials and a device.

```python
ibm = qd.build_backend("IBM_BACKEND", qd.BackendConfig(extra={"backend_name": "ibm_kobe", "pack": True}))
dist = qd.circuit_cutting_sampling(circuit, cuts, backend=ibm, shots=4096, circuits_per_task=15)
```

Under PyCOMPSs each worker connects to IBM again, reading `IBM_QUANTUM_TOKEN` from its environment, so it needs a route to IBM: nodes without internet access, such as MareNostrum 5's, cannot submit these jobs.

This is the end of the numbered notebooks. The **applications** put them together on larger examples.